# Travel Itinerary Agent with File Saving tool

## Overview
This code demonstrates how to create an AI agent that plans travel itineraries by searching the web and saving the results to a markdown file. The agent combines web search capabilities with a custom file-saving tool to produce and persist detailed travel plans.

## Custom Tool Creation

### The `save_to_md` Tool
A custom tool is created using the `@function_tool` decorator that enables the agent to save content to markdown files:

```python
@function_tool
def save_to_md(content: str, filename: str):
```

**How it works:**
1. **Function Definition**: The tool is defined as a regular Python function that accepts two parameters: `content` (the text to save) and `filename` (the output file name)
2. **Decorator**: The `@function_tool` decorator converts this function into a tool that the agent can understand and invoke
3. **Functionality**: 
   - Creates an `output` directory if it doesn't exist
   - Writes the content to a `.md` file in that directory
   - Returns a confirmation message indicating where the file was saved

This decorator-based approach allows the agent to automatically understand the tool's purpose, parameters, and how to use it through the function's signature and docstring.

## Agent Configuration

### Tools Array
The agent is equipped with two tools:
1. **Web Search Tool** (`web_search_tool`): Configured with low context size to minimize API costs while enabling the agent to search for current travel information
2. **Save to Markdown Tool** (`save_to_md`): The custom tool created above for persisting the itinerary

### Agent Instructions
The agent receives clear instructions explaining its role as a trip planner and explicitly directing it to use the save tool to export the final plan in markdown format.

## Execution

The agent is run using `Runner.run()` within a tracing context, which:
1. Processes the trip planning directions
2. Autonomously decides when to use the web search tool to gather information
3. Compiles the information into a detailed itinerary
4. Invokes the `save_to_md` tool to save the complete plan to `output/trip_plan.md`
5. Returns the final output, which is printed to the console

The key insight is that the agent intelligently orchestrates both tools without explicit procedural programming—it determines when to search, what to search for, and when to save the results based on its instructions and the available tools.

In [3]:
import os
from dotenv import load_dotenv, find_dotenv, dotenv_values
from agents import Agent, Runner, Tool, WebSearchTool, trace, function_tool

# Locate .env in this directory or any parent directory
dotenv_path = find_dotenv()
if not dotenv_path:
    raise FileNotFoundError('.env not found in repository or parent directories')

# Load into os.environ (preserves existing variables unless overridden by .env)
load_dotenv(dotenv_path, override=False)
# Also read raw values as a dict (useful to expose into notebook globals)
env = {k: v for k, v in dotenv_values(dotenv_path).items() if v is not None}

# Export into notebook globals for easy access by name
globals().update(env)

print('Loaded .env from', dotenv_path)
print('Loaded keys:', list(env.keys()))

Loaded .env from /media/nathan/linux_ssd/github/agentic_ai_trip_planner/.env
Loaded keys: ['OPENAI_API_KEY', 'GROQ_API_KEY', 'PUSHOVER_USER', 'PUSHOVER_TOKEN', 'SENDGRID_API_KEY', 'GOOGLE_API_KEY', 'SERPER_API_KEY', 'LANGSMITH_TRACING', 'LANGSMITH_ENDPOINT', 'LANGSMITH_API_KEY', 'LANGSMITH_PROJECT', 'POLYGON_API_KEY', 'POLYGON_PLAN', 'BRAVE_API_KEY']


In [ ]:
# Define trip parameters and directions
duration = "5 days"
destination = "Paris"
activities = ["sightseeing", "dining", "arcades", "museums"]
directions = f"plan a {duration} trip to {destination} including these activities: {', '.join(activities)}. Use the web search tool to find relevant information. Produce a detailed itinerary. Save the complete trip plan to a markdown file named 'trip_plan'."

# Define the save to markdown tool
@function_tool
def save_to_md(content: str, filename: str):
    """Save content to a markdown file in the output directory."""
    os.makedirs("output", exist_ok=True)
    with open(f"output/{filename}", "w") as f:
        f.write(content)
    return f"Saved trip plan to output/{filename}"

# Create the web search tool with low context size to limit costs
web_search_tool = WebSearchTool(search_context_size="low") 

# Define the trip planner agent with the two tools.
trip_planner_agent = Agent(
    name="Trip Planner Agent",
    instructions="An agent that helps users plan trips by searching for destinations, accommodations, and activities. Use the save tool to save the final plan in markdown format.",
    tools=[
        web_search_tool,
        save_to_md
    ]
)

# Run the trip planner agent with save to md tool
with trace("Trip Planner Agent"):
    result = await Runner.run(trip_planner_agent, directions)
    print(result.final_output)

Your 5-day Paris trip itinerary has been created and saved as 'trip_plan.md'. Here is a detailed overview:

---

# 5-Day Paris Trip Itinerary

## Day 1: Arrival & Classic Sights
- Morning:
  - Arrive in Paris and check-in at your hotel (recommendation: Hôtel Le Six, centrally located, 4-star, great reviews).
- Afternoon:
  - Visit the Eiffel Tower (advance ticket booking recommended). Take an elevator to the summit for panoramic views.
  - Stroll along the Seine River, visit Champ de Mars.
- Evening:
  - Dinner at Le Ciel de Paris (amazing views, modern French cuisine).

## Day 2: Museums & Arcades
- Morning:
  - Louvre Museum (book tickets in advance, visit the Mona Lisa, Egyptian Antiquities, and more). Arrive early to avoid crowds.
- Afternoon:
  - Explore the Musée d'Orsay (Impressionist art).
- Late Afternoon:
  - Visit Neo-Arcade (56 Rue de la Jonquière, highly rated modern arcade in Paris) for retro and VR games.
- Evening:
  - Dinner at Bistrot Paul Bert (classic Parisian bistr